# Adaptive RAG on Colab

This notebook stands my thesis system up on a fresh Colab GPU runtime and runs
it end to end namely clone the repo, install the CUDA stack, prepare a few
datasets, build an index, smoke-test the pipeline offline, then a small real
evaluation with the 7B generator. The system is a zero-shot LLM router over
three retrieval tiers with a corrective gate that can promote a query one tier
up when the evidence looks wrong.

The runtime is GPU and T4 is enough. Set it before running anything
(Runtime, Change runtime type, choose GPU).


## 1. Environment and GPU check

If the cell below shows no NVIDIA GPU, switch the runtime first and rerun
from the top. T4 is the default and enough for everything here; L4 or A100
only help for the long evaluation sweeps.


In [3]:
import os, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

print(f"python : {sys.version.split()[0]}")
print(f"colab  : {IN_COLAB} | kaggle: {IN_KAGGLE}")

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found -- no NVIDIA GPU visible on this runtime.")

try:
    import torch
    print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  cuda:{i} {torch.cuda.get_device_name(i)}")
        free, total = torch.cuda.mem_get_info()
        print(f"  vram: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
        cap = torch.cuda.get_device_capability(0)
        if cap < (8, 0):
            print(f"  note: compute capability {cap} < (8,0): no native bfloat16 "
                  "(T4/P100). Works, but bf16 ops are emulated and slower.")
except ImportError:
    print("torch not importable yet -- it is installed in the setup cell below.")

if not shutil.which("nvidia-smi"):
    print("\nWARNING: no GPU. The 7B generator would fall back to CPU and be "
          "unusably slow. Switch the runtime to GPU before continuing.")


python : 3.12.13
colab  : True | kaggle: False
Sun Aug 16 21:33:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             29W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+

## 2. Get the repo

The repo is public at github.com/abym-droid/zero-shot-adaptive-corrective-rag,
so the cell below clones it and moves into it. All repo-relative paths
(data/, indices/, runs/) resolve from there.


In [4]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/abym-droid/zero-shot-adaptive-corrective-rag.git"
WORK_DIR = Path("/content")
REPO_DIR = WORK_DIR / "zero-shot-adaptive-corrective-rag"

if (REPO_DIR / "pyproject.toml").exists():
    print(f"repo already present: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

import os
os.chdir(REPO_DIR)  # all repo-relative paths (data/, indices/, runs/) resolve from here
print("cwd:", os.getcwd())


cwd: /content/zero-shot-adaptive-corrective-rag


## 3. Install the CUDA dependency stack

Same as `environment/requirements-core.txt` minus the Apple-only packages,
namely mlx and mlx-lm. Generation goes through the HF backend, which picks up
bitsandbytes 4-bit on CUDA, so the model runs at the same reduced precision
as the MLX path on my laptop. Torch ships preinstalled with CUDA on Colab
and is left alone.


In [5]:
%pip install -q "transformers>=4.56" "accelerate>=0.33" "sentence-transformers>=3.0" \
    "datasets>=2.20" "huggingface_hub>=0.24" "faiss-cpu>=1.8" "rank-bm25>=0.2.2" \
    "langgraph>=0.2" "langchain-core>=0.2" "pydantic>=2.7" "pydantic-settings>=2.3" \
    "typer>=0.12" rich tqdm jsonlines orjson pyyaml evaluate rouge-score \
    "bitsandbytes>=0.43"
%pip install -q -e .


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 60.9 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adarag (pyproject.toml) ... done


In [6]:
# Import probe, the editable install must resolve in this kernel without a restart.
import sys
from pathlib import Path

src = str(Path("src").resolve())
if src not in sys.path:           # belt-and-braces for editable installs
    sys.path.insert(0, src)

from adarag.config import settings
from adarag.device import get_device

print("adarag import OK")
print("resolved device      :", get_device(settings.device))   # expect: cuda
print("max_escalations bound:", settings.max_escalations, "(one tier promotion max)")

# Environment / settings / model-cache report from the CLI entry point:
!adarag info


adarag import OK
resolved device      : cuda
max_escalations bound: 1 (one tier promotion max)
                          environment                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ component             ┃ value                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ python                │ 3.12.13 (/usr/bin/python3)           │
│ platform              │ Linux-6.6.122+-x86_64-with-glibc2.35 │
│ adarag                │ 0.0.1                                │
│ torch                 │ 2.11.0+cu128                         │
│ transformers          │ 5.13.1                               │
│ sentence-transformers │ 5.6.0                                │
│ mlx-lm                │ not installed                        │
│ langgraph             │ 1.2.9                                │
│ langchain-core        │ 1.4.9                                │
│ faiss-cpu             │ 1.15.0                            

### 3a. Optional Hugging Face login

Only needed for the gated telecom sources (teleqna, tspec-llm) after
accepting their terms on the Hub. Everything else in this notebook runs
without a token. The cell prompts for the token with getpass, so it never
appears in the notebook or its output. Set RUN_HF_LOGIN to True to use it.


In [7]:
RUN_HF_LOGIN = True  # set True when the gated telecom sources are needed

if RUN_HF_LOGIN:
    import os
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("HF token: ")
    !hf download rasoul-nikbakht/TSpec-LLM --repo-type dataset --include "3GPP-clean/Rel-16/*.md" --include "3GPP-clean/Rel-17/*.md" --local-dir data/raw/tspec_llm
else:
    print("skipped (RUN_HF_LOGIN = False)")


Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.23.0.
To update, run: hf update
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3568 files:   0% 0/3568 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/181k [00:00<?, ?B/s]          
Reconstructing (incomplete total...):   0% 0.00/213k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/213k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/240k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/251k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/251k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/312k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/324k [00:00<?, ?B/s]
Reconstructing (incomplete total...):  15% 181k/1.19M [00:00<00:04, 217kB/s, 18.0kB/s  ]
Reconstructing (incomplete total...):  16% 197k/1.25M [00:00<00:04, 217kB/s,   497B/s  ]
Reconstructi

## 4. Get the datasets

Raw files are fetched straight from their sources into `data/raw/`, then
`scripts/prepare_datasets.py` normalises them and cuts the seeded dev500 and
toy slices. Same script and seed as on my laptop, so the slices match
exactly.

Here I take squad (single-hop), hotpotqa (multi-hop, gives the router a
reason to pick the iterative tier) and scifact, which also provides the
retrieval corpus indexed in the next step.


In [9]:
%%bash
mkdir -p data/raw/squad data/raw/hotpotqa data/raw/scifact
H=https://huggingface.co/datasets
wget -nc -q -O data/raw/squad/validation.parquet    $H/rajpurkar/squad/resolve/main/plain_text/validation-00000-of-00001.parquet
wget -nc -q -O data/raw/hotpotqa/validation.parquet $H/hotpotqa/hotpot_qa/resolve/main/distractor/validation-00000-of-00001.parquet
wget -nc -q -O data/raw/scifact/claims.parquet      $H/allenai/scifact/resolve/refs%2Fconvert%2Fparquet/claims/validation/0000.parquet
wget -nc -q -O data/raw/scifact/corpus.parquet      $H/allenai/scifact/resolve/refs%2Fconvert%2Fparquet/corpus/train/0000.parquet
python scripts/prepare_datasets.py squad hotpotqa scifact
echo; echo '--- processed files ---'; ls -la data/processed/


[squad] 10570 rows (dev500: 500, toy: 20)
[squad] 2067 corpus docs -> squad.corpus.jsonl
[hotpotqa] 7405 rows (dev500: 500, toy: 20)
[hotpotqa] 66635 corpus docs -> hotpotqa.corpus.jsonl
[scifact] 450 rows (dev500: 450, toy: 20)
[scifact] 5183 corpus docs -> scifact.corpus.jsonl

--- processed files ---
total 63240
drwxr-xr-x 2 root root     4096 Aug 17 00:01 .
drwxr-xr-x 4 root root     4096 Aug 17 00:01 ..
-rw-r--r-- 1 root root 40136150 Aug 17 00:13 hotpotqa.corpus.jsonl
-rw-r--r-- 1 root root   172731 Aug 17 00:13 hotpotqa.dev500.jsonl
-rw-r--r-- 1 root root  2553562 Aug 17 00:13 hotpotqa.jsonl
-rw-r--r-- 1 root root     6734 Aug 17 00:13 hotpotqa.toy.jsonl
-rw-r--r-- 1 root root  8038622 Aug 17 00:13 scifact.corpus.jsonl
-rw-r--r-- 1 root root   169325 Aug 17 00:13 scifact.dev500.jsonl
-rw-r--r-- 1 root root   169325 Aug 17 00:13 scifact.jsonl
-rw-r--r-- 1 root root     7587 Aug 17 00:13 scifact.toy.jsonl
-rw-r--r-- 1 root root  1728260 Aug 17 00:13 squad.corpus.jsonl
-rw-r--r-- 1

## 5. Build retrieval indices

`adarag index` chunks a corpus jsonl and writes a BM25 index and a FAISS
dense index (MiniLM embeddings, encoded on the GPU here). Two indices are
built namely a tiny inline telecom corpus for the offline smoke test, and
the real SciFact corpus for the evaluation run.


In [10]:
import json
from pathlib import Path

TOY_DOCS = [
    {"doc_id": "toy-001", "title": "3GPP and 5G NR",
     "text": "The 3rd Generation Partnership Project (3GPP) publishes the 5G New Radio "
             "(NR) specifications. 5G NR was first standardised in 3GPP Release 15, with "
             "further enhancements in Release 16 and Release 17."},
    {"doc_id": "toy-002", "title": "OSS and BSS",
     "text": "In telecom operations, OSS (Operations Support Systems) manage network "
             "inventory, provisioning and fault management, while BSS (Business Support "
             "Systems) handle billing, CRM and order management."},
    {"doc_id": "toy-003", "title": "Retrieval-Augmented Generation",
     "text": "Retrieval-Augmented Generation (RAG) grounds a language model's answer in "
             "passages fetched from an external corpus, reducing hallucination on "
             "knowledge-intensive questions."},
    {"doc_id": "toy-004", "title": "BM25",
     "text": "BM25 is a sparse lexical ranking function based on term frequency, inverse "
             "document frequency and document length normalisation. It remains a strong "
             "retrieval baseline."},
    {"doc_id": "toy-005", "title": "Dense retrieval",
     "text": "Dense retrieval encodes queries and passages into a shared embedding space "
             "and ranks passages by vector similarity, typically with a FAISS index."},
    {"doc_id": "toy-006", "title": "Adaptive-RAG",
     "text": "Adaptive-RAG routes each query to a retrieval strategy matched to its "
             "complexity: no retrieval, single-step retrieval, or iterative multi-step "
             "retrieval."},
    {"doc_id": "toy-007", "title": "CRAG",
     "text": "Corrective RAG (CRAG) evaluates the quality of retrieved evidence and "
             "triggers a corrective action when the evidence is judged incorrect."},
    {"doc_id": "toy-008", "title": "Network slicing",
     "text": "5G network slicing partitions one physical network into multiple virtual "
             "end-to-end networks, each tailored to a service class such as eMBB, URLLC "
             "or mMTC."},
    {"doc_id": "toy-009", "title": "ColBERTv2",
     "text": "ColBERTv2 is a late-interaction retriever: it keeps token-level embeddings "
             "and computes MaxSim interactions between query and document tokens at "
             "search time."},
    {"doc_id": "toy-010", "title": "IRCoT",
     "text": "IRCoT interleaves retrieval with chain-of-thought reasoning: each reasoning "
             "step issues a new retrieval query, supporting multi-hop questions."},
]

Path("data/toy").mkdir(parents=True, exist_ok=True)
with open("data/toy/corpus.jsonl", "w", encoding="utf-8") as fh:
    for d in TOY_DOCS:
        fh.write(json.dumps(d, ensure_ascii=False) + "\n")
print(f"wrote {len(TOY_DOCS)} toy docs -> data/toy/corpus.jsonl")

!adarag index data/toy/corpus.jsonl --out-dir data/toy/indices


wrote 10 toy docs -> data/toy/corpus.jsonl
building ('bm25', 'dense') indices for data/toy/corpus.jsonl -> data/toy/indices
modules.json: 100% 349/349 [00:00<00:00, 2.10MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 886kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 5.74MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 415kB/s]
config.json: 100% 612/612 [00:00<00:00, 4.63MB/s]

model.safetensors: downloading bytes:  74% 66.8M/90.9M [00:00<00:00, 98.3MB/s, 5.15MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 81.7MB/s, 8.17MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 87.3MB/s, 8.92MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 4506.15it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.59MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 8.96MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 8.72MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 810kB/s]
config.json: 100% 190/190 [0

In [11]:
# SciFact corpus index (~5k abstracts; MiniLM encoding runs on the GPU).
!adarag index data/processed/scifact.corpus.jsonl --out-dir indices/scifact

print("\n--- index layout ---")
!find indices/scifact -maxdepth 2 -type f | sort


building ('bm25', 'dense') indices for data/processed/scifact.corpus.jsonl -> 
indices/scifact
Loading weights: 100% 103/103 [00:00<00:00, 4688.41it/s]
done

--- index layout ---
indices/scifact/bm25/bm25.pkl
indices/scifact/dense/chunks.jsonl
indices/scifact/dense/index.faiss
indices/scifact/dense/meta.json
indices/scifact/meta.json


## 6. Pipeline smoke test, offline

Before spending GPU time I check the whole loop, route then execute then
gate then escalate, with the deterministic fake backend. No model downloads,
no GPU. This is the same offline path the test suite uses.


In [12]:
!adarag ask "Which organisation publishes the 5G NR specifications?" --fake

# Harness smoke: 10 SQuAD dev500 examples through the eval loop on canned responses.
!adarag eval --dataset data/processed/squad.dev500.jsonl --backend fake --limit 10 \
    --out-dir runs/fake_smoke


╭─────────────────────────────────── answer ───────────────────────────────────╮
│ This is a deterministic demo answer produced by FakeBackend.                 │
╰──────────────────────────────────────────────────────────────────────────────╯
 tier (final)    single_step                        
 route reason    FakeBackend canned decision (demo) 
 gate verdict    correct                            
 gate reason     FakeBackend canned verdict (demo)  
 escalations     0                                  
 docs retrieved  3                                  
                          trace                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ node         ┃ latency_s ┃ prompt_tok ┃ completion_tok ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ route        │     0.002 │        333 │              7 │
│ execute_tier │     0.000 │        145 │              9 │
│ gate         │     0.001 │        342 │              9 │
└──────────────┴──────────

## 7. Real evaluation run

Now the real thing namely `adarag eval` over the SciFact dev500 slice with
the BM25 index and the HF backend (Qwen2.5-7B-Instruct in 4-bit, about
5.5 GB VRAM, fits a T4). Each example records the routing decision, gate
verdict, escalations, EM/F1, latency and token counts; the run directory
gets predictions.jsonl and summary.json.

A few notes.

* the first run downloads about 15 GB of model weights, cached afterwards
* the run here is capped with `--limit 20`; a thesis-grade run drops the
  limit over the full dev500
* the ablation flags are `--no-escalation`, `--prompt-variant v2` and
  `--retriever dense`


In [13]:
RUN_DIR = "runs/colab_scifact_hf"

!adarag eval --dataset data/processed/scifact.dev500.jsonl --index indices/scifact \
    --backend hf --retriever bm25 --prompt-variant v1 --limit 20 \
    --out-dir $RUN_DIR


config.json: 100% 663/663 [00:00<00:00, 4.53MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 19.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 12.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 13.2MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 19.3MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 654kB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.50G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/11.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   3% 502M/15.2G [00:01<00:37, 390MB/s, 17.7MB/s  ]
Reconstructing (incomplete total...):  13% 2.01G/15.2G [00:06<01:01, 216MB/s, 46.5MB/s  ]
Reconstructing (incomplete total...):  41% 6.25G/15.2G [00:18<00:25, 353MB/s, 34.7MB/s  ]
Reconstructing (incomplete total...):  75

In [14]:
import json
from pathlib import Path

run_dir = Path(RUN_DIR)
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
print("=== summary.json ===")
print(json.dumps(summary, indent=2))

# Per-example view: routing tiers, escalations, gate verdicts, scores.
import pandas as pd

rows = [json.loads(l) for l in (run_dir / "predictions.jsonl").read_text(
    encoding="utf-8").splitlines() if l.strip()]
df = pd.DataFrame(rows)[
    ["qid", "tier_initial", "tier_final", "escalated", "verdict",
     "em", "f1", "latency_s", "prompt_tokens", "completion_tokens"]
]
display(df.head(20))
print("\ntier distribution (initial):", df["tier_initial"].value_counts().to_dict())
print("escalation rate            :", round(float(df["escalated"].mean()), 3))


=== summary.json ===
{
  "n_examples": 20,
  "em": 0.65,
  "f1": 0.0,
  "mcq_accuracy": 0.65,
  "n_mcq": 20,
  "escalation_rate": 0.55,
  "tier_initial_counts": {
    "single_step": 18,
    "no_retrieval": 2
  },
  "tier_final_counts": {
    "iterative": 10,
    "single_step": 9,
    "no_retrieval": 1
  },
  "verdict_counts": {
    "incorrect": 11,
    "correct": 9
  },
  "mean_latency_s": 101.16038499999999,
  "mean_prompt_tokens": 12660.8,
  "mean_completion_tokens": 115.65,
  "total_tokens": 255529,
  "routing_accuracy_vs_silver": 0.8666666666666667,
  "config": {
    "dataset": "data/processed/scifact.dev500.jsonl",
    "index": "indices/scifact",
    "backend": "hf",
    "model": null,
    "retriever": "bm25",
    "prompt_variant": "v1",
    "escalation": true,
    "max_escalations": 1,
    "top_k": 5,
    "limit": 20,
    "device": "auto",
    "timestamp": "20260817_001639"
  }
}


,qid,tier_initial,tier_final,escalated,verdict,em,f1,latency_s,prompt_tokens,completion_tokens
0,scifact-1,single_step,iterative,True,incorrect,1.0,0.0,190.1781,24684,137
1,scifact-3,single_step,iterative,True,incorrect,0.0,0.0,156.5114,20415,132
2,scifact-3,single_step,iterative,True,incorrect,0.0,0.0,156.1460,20415,132
3,scifact-5,single_step,single_step,False,correct,0.0,0.0,38.4608,4329,90
4,scifact-13,no_retrieval,single_step,True,incorrect,0.0,0.0,56.3829,6351,130
5,scifact-36,no_retrieval,no_retrieval,False,correct,0.0,0.0,13.2251,1199,55
6,scifact-42,single_step,iterative,True,incorrect,1.0,0.0,190.7408,24366,161
7,scifact-42,single_step,iterative,True,incorrect,1.0,0.0,190.4181,24366,161
8,scifact-48,single_step,iterative,True,incorrect,1.0,0.0,79.0405,9273,153
9,scifact-49,single_step,single_step,False,correct,1.0,0.0,33.9419,3934,75



tier distribution (initial): {'single_step': 18, 'no_retrieval': 2}
escalation rate            : 0.55


## 8. Next steps

* run the full dev500 files with no `--limit`, one run directory per dataset
* work through the ablation grid namely escalation on and off, prompt
  variant v1 and v2, bm25 and dense
* for the telecom domain accept the gated terms on the Hub, run cell 3a,
  download, then index the tspec corpus
* routing accuracy against silver labels is already in summary.json
* copy results off the VM before it recycles, either zip the runs directory
  and download it or mount Drive and copy it there
